In [ ]:
import torch
import pandas as pd
import json
import time
import datetime
import shutil
import os
import gc
import traceback
import argparse
import random
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import re
from nltk import ngrams
from collections import Counter
import matplotlib.pyplot as plt
import copy
import seaborn as sns
import numpy as np
import torch.nn.functional as F
import typing
import agentdojo
from typing import Optional


import utils.attack_utility as attack_utility
import utils.experiment_logger as experiment_logger
from secalign_refactored import secalign, config
import algorithms.losses_experimental as losses_experimental
import pandas as pd

from agentdojo.task_suite.load_suites import get_suites
from agentdojo.agent_pipeline import GroundTruthPipeline
from agentdojo.functions_runtime import FunctionsRuntime
import warnings, yaml
from pathlib import Path


In [ ]:
suites = get_suites("v1.2")
suite_data = {}

for suite_name, suite in suites.items():
    # Find injection vector names from the YAML
    vectors = suite.get_injection_vector_defaults()

    # Inject unique markers
    MARKER = "___MARKER___"
    marker_injections = {v: f"{MARKER}{v}{MARKER}" for v in vectors}

    injectable_ut_ids = set()

    for ut_id, user_task in suite.user_tasks.items():
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                env = suite.load_and_inject_default_environment(marker_injections)
                if hasattr(user_task, "init_environment"):
                    env = user_task.init_environment(env)
                runtime = FunctionsRuntime(suite.tools)
                gt = GroundTruthPipeline(user_task)
                _, _, _, messages, _ = gt.query(user_task.PROMPT, runtime, env)

            if any(f"{MARKER}{v}{MARKER}" in str(messages) for v in vectors):
                injectable_ut_ids.add(ut_id)
        except Exception as e:
            print(f"  ⚠ {suite_name}/{ut_id}: {e}")
            injectable_ut_ids.add(ut_id)  # conservative

    suite_data[suite_name] = {
        "suite": suite,
        "injectable_user_tasks": injectable_ut_ids,
    }
    print(f"{suite_name}: {len(injectable_ut_ids)}/{len(suite.user_tasks)} user tasks injectable, "
          f"{len(suite.injection_tasks)} injection tasks "
          f"→ {len(injectable_ut_ids) * len(suite.injection_tasks)} valid combos")

In [ ]:
import pandas as pd

rows = []
for suite_name, sd in suite_data.items():
    suite = sd["suite"]
    inj_set = sd["injectable_user_tasks"]
    for ut_id, ut in suite.user_tasks.items():
        if ut_id not in inj_set:
            continue
        for it_id, it in suite.injection_tasks.items():
            rows.append({
                "suite": suite_name,
                "user_task_id": ut_id,
                "user_task_prompt": ut.PROMPT,
                "injection_task_id": it_id,
                "injection_task_goal": it.GOAL,
            })

combos = pd.DataFrame(rows)
combos

In [ ]:

flattened_agentdojo_successes = []
with open("data/agentdojo_data/agentdojo_successes.json", "r") as agentdojo_successes_file:
    agentdojo_successes = json.load(agentdojo_successes_file)

for agentdojo_success in agentdojo_successes:
    flattened_agentdojo_successes += agentdojo_success["prompt_injection"]

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

# Build corpus from injection tasks only (deduplicated)
inj_corpus = combos[["suite", "injection_task_id", "injection_task_goal"]].drop_duplicates()
corpus_texts = inj_corpus["injection_task_goal"].tolist()
corpus_labels = (inj_corpus["suite"] + "/" + inj_corpus["injection_task_id"]).tolist()

q_emb = model.encode(flattened_agentdojo_successes, normalize_embeddings=True, show_progress_bar=True)
c_emb = model.encode(corpus_texts, normalize_embeddings=True, show_progress_bar=True)

sims = q_emb @ c_emb.T  # (n_successes, n_corpus)

TOP_K = 3

results = []
for i, inj_text in enumerate(flattened_agentdojo_successes):
    top_idx = sims[i].argsort()[::-1][:TOP_K]
    results.append({
        "injection": inj_text,
        "best_match_label": corpus_labels[top_idx[0]],
        "best_match_goal": corpus_texts[top_idx[0]],
        "best_score": float(sims[i, top_idx[0]]),
        "top_k": [
            {"label": corpus_labels[j], "goal": corpus_texts[j], "score": float(sims[i, j])}
            for j in top_idx
        ],
    })

results_df = pd.DataFrame(results).drop(columns=["top_k"])
results_df

In [ ]:
merged = results_df.merge(
    combos,
    left_on="best_match_label",
    right_on=combos["suite"] + "/" + combos["injection_task_id"],
    how="left",
)
# Now each row has: injection, best_match, score, AND the suite/user_task/tools context
merged[["injection", "best_match_label", "best_score", "suite",
        "user_task_id", "user_task_prompt"]].head(10)

In [ ]:
results_df.to_csv("data/agentdojo_data/matched_results.csv", index=False)
merged.to_csv("data/agentdojo_data/matched_with_combos.csv", index=False)